# 01 · Conformal prediction intervals

Add distribution-free prediction intervals to point forecasts using
split conformal calibration on CV residuals.

**Procedure**

1. Load a rolling-origin CV artifact (`artifacts/cv_<model>.parquet`).
2. Compute residuals per series × horizon step × CV window.
3. Fit a `ConformalCalibrator` (pooled quantile, grouped, or scaled).
4. Apply intervals to a held-out or future forecast.
5. Diagnose coverage, interval width, and the α–width trade-off.

**Inputs**
- `data/processed/long.parquet` — produced by `make prep`.
- `artifacts/cv_<model>.parquet` — rolling-origin CV output.
- `forecasts/forecast_<model>.parquet` — future forecast (optional).

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from __future__ import annotations
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from m5.config import SETTINGS
from m5.conformal import ConformalCalibrator, compute_residuals, coverage, average_interval_width
from m5.plots import configure_style

configure_style()

ARTIFACTS = SETTINGS.artifacts_dir
FORECASTS = SETTINGS.forecasts_dir
LONG_PARQUET = SETTINGS.processed_dir / 'long.parquet'

In [ ]:
# Pick a model artifact
MODEL = "lgbm"  # change to "stats", "hier", etc.

cv_path = ARTIFACTS / f"cv_{MODEL}.parquet"
fc_path = FORECASTS / f"forecast_{MODEL}.parquet"

if not cv_path.exists():
    raise FileNotFoundError(f"CV artifact not found: {cv_path} — run `make cv-{MODEL}` first.")

cv = pd.read_parquet(cv_path)
cv["ds"] = pd.to_datetime(cv["ds"])
cv["cutoff"] = pd.to_datetime(cv["cutoff"])
print(f"CV frame: {len(cv):,d} rows, {cv['unique_id'].nunique():,d} series, "
      f"{cv['cutoff'].nunique()} windows")

model_cols = [c for c in cv.columns if c not in ("unique_id", "ds", "cutoff", "y")]
print(f"Forecast columns: {model_cols}")

## 1 · Fit calibrator (pooled method)

In [ ]:
# Fit a pooled calibrator — the simplest method
cal = ConformalCalibrator(alpha=0.1, method="pooled")
cal.fit(cv, model_col=model_cols[0])
print(f"Calibrated for {cal.model_col!r}, horizon={cal.horizon}, alpha={cal.alpha}")
print(f"Per-step quantiles:\n{cal._pooled_quantiles}")

## 2 · Apply intervals to the CV frame itself (sanity check)

In [ ]:
with_int = cal.predict(cv)
print(f"Frame shape: {with_int.shape}")
print(f"New columns: {[c for c in with_int.columns if 'lo' in c or 'hi' in c]}")
with_int.head()

In [ ]:
# Compute empirical coverage on the CV data
cv_coverage = coverage(
    truth=cv,
    forecast=with_int,
    lo_col=f"{model_cols[0]}_lo",
    hi_col=f"{model_cols[0]}_hi",
)
print(f"Empirical coverage on CV: {cv_coverage:.4f}  (target: {1 - cal.alpha:.4f})")

avg_width = average_interval_width(
    forecast=with_int,
    lo_col=f"{model_cols[0]}_lo",
    hi_col=f"{model_cols[0]}_hi",
)
print(f"Average interval width: {avg_width:.4f}")

## 3 · Coverage by horizon step

In [ ]:
with_int["_step"] = (with_int["ds"] - with_int["cutoff"]).dt.days
lo = f"{model_cols[0]}_lo"
hi = f"{model_cols[0]}_hi"

cov_by_step = (
    with_int.groupby("_step")
    .apply(lambda g: ((g["y"] >= g[lo]) & (g["y"] <= g[hi])).mean())
    .rename("coverage")
)
width_by_step = (
    with_int.groupby("_step")
    .apply(lambda g: (g[hi] - g[lo]).mean())
    .rename("avg_width")
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
cov_by_step.plot(ax=ax1, marker="o")
ax1.axhline(1 - cal.alpha, color="red", linestyle="--", label=f"target {1-cal.alpha}")
ax1.set_title(f"Coverage by horizon step ({cal.model_col})")
ax1.set_xlabel("Horizon step")
ax1.set_ylabel("Coverage")
ax1.legend()

width_by_step.plot(ax=ax2, marker="o", color="orange")
ax2.set_title(f"Average interval width by horizon step ({cal.model_col})")
ax2.set_xlabel("Horizon step")
ax2.set_ylabel("Width")
fig.tight_layout()

## 4 · Compare calibration methods

In [ ]:
# Try all three methods and compare
results = {}
for method in ("pooled", "scaled"):
    c = ConformalCalibrator(alpha=0.1, method=method)
    c.fit(cv, model_col=model_cols[0])
    pred = c.predict(cv)
    lo_col = f"{model_cols[0]}_lo"
    hi_col = f"{model_cols[0]}_hi"
    results[method] = {
        "coverage": coverage(cv, pred, lo_col=lo_col, hi_col=hi_col),
        "avg_width": average_interval_width(pred, lo_col=lo_col, hi_col=hi_col),
    }

pd.DataFrame(results).T

## 5 · α–width trade-off sweep

In [ ]:
alphas = np.linspace(0.01, 0.5, 20)
sweep = []
for a in alphas:
    c = ConformalCalibrator(alpha=a, method="pooled")
    c.fit(cv, model_col=model_cols[0])
    pred = c.predict(cv)
    lo_col = f"{model_cols[0]}_lo"
    hi_col = f"{model_cols[0]}_hi"
    sweep.append({
        "alpha": a,
        "coverage": coverage(cv, pred, lo_col=lo_col, hi_col=hi_col),
        "avg_width": average_interval_width(pred, lo_col=lo_col, hi_col=hi_col),
    })
sweep_df = pd.DataFrame(sweep)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(sweep_df["alpha"], sweep_df["coverage"], marker="o")
ax1.plot(sweep_df["alpha"], 1 - sweep_df["alpha"], "--", color="red", label="nominal")
ax1.set_xlabel("α")
ax1.set_ylabel("Empirical coverage")
ax1.legend()
ax2.plot(sweep_df["alpha"], sweep_df["avg_width"], marker="o", color="orange")
ax2.set_xlabel("α")
ax2.set_ylabel("Average interval width")
fig.tight_layout()

## 6 · Apply to a future forecast

In [ ]:
if fc_path.exists():
    fc_df = pd.read_parquet(fc_path)
    fc_df["ds"] = pd.to_datetime(fc_df["ds"])
    print(f"Forecast frame: {len(fc_df):,d} rows")

    fc_with_int = cal.predict(fc_df)
    print(fc_with_int.head())
else:
    print(f"No forecast artifact at {fc_path}. Run `make forecast-{MODEL}` first.")
    fc_with_int = None

## 7 · Visualise intervals for a few series

In [ ]:
if fc_with_int is not None:
    lo_col = f"{model_cols[0]}_lo"
    hi_col = f"{model_cols[0]}_hi"

    # Pick a few series to plot
    sample_ids = cv["unique_id"].drop_duplicates().sample(4, random_state=42)
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    for ax, sid in zip(axes.ravel(), sample_ids, strict=False):
        sub = fc_with_int[fc_with_int["unique_id"] == sid].sort_values("ds")
        ax.plot(sub["ds"], sub[model_cols[0]], "b-", label="forecast")
        ax.fill_between(sub["ds"], sub[lo_col], sub[hi_col], alpha=0.3, color="blue", label=f"{int((1-cal.alpha)*100)}% CI")
        ax.set_title(sid)
        ax.legend(fontsize=8)
    fig.tight_layout()

## 8 · Diagnostics: residual distribution

In [ ]:
res = compute_residuals(cv, model_col=model_cols[0])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Clip each histogram's x-axis to mean ± 3σ of its own data
r = res["residual"]
lo0, hi0 = r.mean() - 3 * r.std(), r.mean() + 3 * r.std()
axes[0].hist(r, bins=100, edgecolor="none")
axes[0].set_xlim(lo0, hi0)
axes[0].set_xlabel("residual (y − ŷ)")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Residual distribution (pooled)")
axes[0].axvline(r.median(), color="red", ls="--", lw=1, label=f"median={r.median():.2f}")
axes[0].legend(fontsize=8)

ra = r.abs()
lo1, hi1 = 0.0, ra.mean() + 3 * ra.std()
axes[1].hist(ra, bins=100, edgecolor="none")
axes[1].set_xlim(lo1, hi1)
axes[1].set_xlabel("|residual|")
axes[1].set_title("Absolute residual distribution")

for step in range(1, 29, 7):
    sub = res[res["step"] == step]["residual"]
    axes[2].hist(sub, bins=50, alpha=0.5, label=f"step={step}")
# Align x-axis of last panel with the pooled residual range
axes[2].set_xlim(lo0, hi0)
axes[2].set_xlabel("residual")
axes[2].set_title("Residual by horizon step")
axes[2].legend(fontsize=8)
fig.tight_layout()